In [1]:
#Set up the environment and imports
import os
import sys
from dotenv import load_dotenv
from langfuse.openai import openai
from langfuse import observe, get_client

# Add the repo root to path so we can import from shared/
sys.path.append(os.path.abspath(".."))
from shared.shopbot_config import SHOPBOT_SYSTEM_PROMPT

load_dotenv("../.env")

client = openai.OpenAI(
    api_key=os.getenv("GOOGLE_AI_API"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "gemini-3.5-flash-lite"

# get_client() hands back the same Langfuse client the decorator uses
# internally - this is how we attach scores after the pipeline returns
langfuse = get_client()

In [2]:
POLICY_SNIPPETS = {
    "shipping": "Free shipping is available on all orders over $75.",
    "loyalty": "The StyleHive Rewards loyalty program is open to all registered members.",
    "support hours": "Customer support is available Monday to Friday, 9am-6pm EST.",
    "30-day return": "StyleHive accepts returns within 30 days of delivery for unworn items with tags attached.",
}

#Decorating the retrieval step so it shows up as its own span
@observe()
def retrieve_policy(question: str) -> str:
    for keyword, snippet in POLICY_SNIPPETS.items():
        if keyword in question.lower():
            return snippet
    return "No specific policy found."

#Decorating the generation step so it shows up as its own span
@observe()
def generate_response(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SHOPBOT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

#Returning context and trace_id alongside the answer - the judge needs
#context, and create_score needs the trace_id
@observe(name="ask-shopbot-pipeline")
def ask_shopbot_pipeline(question: str) -> tuple[str, str, str]:
    context = retrieve_policy(question)
    answer = generate_response(question, context)
    trace_id = langfuse.get_current_trace_id()
    return answer, context, trace_id

In [3]:
question = "Do you accept returns on Black Friday sale items?"
answer, context, trace_id = ask_shopbot_pipeline(question)

print(answer)

Hello! Thanks for reaching out to StyleHive. 

I'm sorry, but I don't have that specific information regarding returns on Black Friday sale items in my current system. Generally, our standard policy applies to unworn items with tags attached within 30 days, but for Black Friday specifics, I don't want to guess. 

If you'd like, you can reach out to our customer support team directly during our business hours (Monday–Friday, 9 AM–6 PM EST) for clarification on Black Friday items! Let me know if I can help you with anything else.


In [4]:
langfuse.create_score(
    trace_id=trace_id,
    name="user-feedback",
    value="negative",
    data_type="CATEGORICAL",
    comment="Customer said this answer was wrong.",
)

In [5]:
JUDGE_PROMPT = """You are grading whether a customer support answer is actually supported by the retrieved context, not by general knowledge.

Question: {question}
Retrieved context: {context}
Answer given: {answer}

Score how well the answer is supported by the retrieved context, from 0.0 (not supported at all, the answer is guessed) to 1.0 (fully supported by the context).

Respond in exactly this format, nothing else:
SCORE: <number between 0.0 and 1.0>
REASON: <one sentence>"""

@observe(name="judge-answer-quality")
def judge_answer_quality(question: str, context: str, answer: str) -> tuple[float, str]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": JUDGE_PROMPT.format(question=question, context=context, answer=answer)},
        ],
    )
    lines = response.choices[0].message.content.strip().splitlines()
    score = float(lines[0].split(":", 1)[1].strip())
    reason = lines[1].split(":", 1)[1].strip()
    return score, reason

In [6]:
score, reason = judge_answer_quality(question=question, context=context, answer=answer)

langfuse.create_score(
    trace_id=trace_id,
    name="answer-quality",
    value=score,
    data_type="NUMERIC",
    comment=reason,
)

print(f"score: {score}")
print(f"reason: {reason}")

score: 1.0
reason: The answer correctly acknowledges that the retrieved context contains no specific policy and avoids guessing while offering a way to get help.
